# Numerisk derivasjon

```{admonition} Læringsutbytte
Etter å ha arbeidet med denne delen av emnet, skal du kunne:

1. forklare forskjellen på analytisk og numerisk derivasjon
2. implementere framover-, bakover- og sentraldifferansen
3. undersøke hvordan steglengde og avrundingsfeil påvirker resultatet
4. derivere eksperimentelle data og tolke hva den numeriske deriverte betyr kjemisk
```

## Derivasjonsbegrepet

Fra videregående kjenner du gjerne den deriverte som $f'(x)$. En annen vanlig notasjon er

$$\frac{df}{dx}.$$

Dette er **Leibniz-notasjon** og betyr «endringen i $f$ med hensyn på $x$». I kjemi er denne notasjonen spesielt nyttig fordi variablene ofte har en fysisk betydning:

$$\frac{dc}{dt}$$

er endringen i konsentrasjon med tid, mens

$$\frac{d\mathrm{pH}}{dV}$$

er endringen i pH med tilsatt volum.

De to notasjonene beskriver det samme når $f$ er en funksjon av $x$:

$$f'(x)=\frac{df}{dx}.$$


Den deriverte er definert som en grenseverdi:

$$f'(x)=\frac{df}{dx}=\lim_{\Delta x\rightarrow 0}\frac{f(x+\Delta x)-f(x)}{\Delta x}.$$

På en datamaskin kan vi ikke bruke et uendelig lite $\Delta x$. Vi velger derfor et lite, men endelig steg $h$:

$$f'(x)\approx\frac{f(x+h)-f(x)}{h}.$$

Dette kalles **framoverdifferansen**.


```{admonition} Underveisoppgave
:class: tip
Beregn $f'(1)$ numerisk for $f(x)=2x+2$ med $h=10^{-8}$. Hva forventer du fra analytisk derivasjon?
```


In [1]:
def f(x):
    return 2*x + 2

x = 1.0
h = 1e-8

f_derivert = (f(x + h) - f(x)) / h
print("Numerisk:", f_derivert)
print("Analytisk:", 2.0)


Numerisk: 1.999999987845058
Analytisk: 2.0


Vi kan pakke framoverdifferansen inn i en funksjon:


In [2]:
def deriver_framover(f, x, h=1e-8):
    return (f(x + h) - f(x)) / h


## Feilanalyse: mindre steg er ikke alltid bedre

Det er fristende å tenke at $h$ bør være så liten som mulig. To ulike feilmekanismer konkurrerer imidlertid:

- **Tilnærmingsfeilen** blir vanligvis mindre når $h$ reduseres.
- **Avrundingsfeil i flyttall** kan bli viktig når vi trekker fra to nesten like tall og deler på et svært lite tall.

Det finnes derfor ikke én universell optimal verdi av $h$. Den avhenger blant annet av funksjonen, tallskalaen og differansemetoden.


In [3]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return 2*x**2 + x - 5

def f_derivert_analytisk(x):
    return 4*x + 1

x0 = 1.0
h_verdier = np.logspace(-1, -16, 16)
eksakt = f_derivert_analytisk(x0)

feil = []
for h in h_verdier:
    numerisk = deriver_framover(f, x0, h)
    feil.append(abs(numerisk - eksakt))

plt.loglog(h_verdier, feil, "o-")
plt.xlabel("h")
plt.ylabel("Absolutt feil")
plt.show()


## Framover-, bakover- og sentraldifferansen

Framoverdifferansen bruker punktet foran $x$:

$$\frac{df}{dx}\approx\frac{f(x+h)-f(x)}{h}.$$

Bakoverdifferansen bruker punktet bak $x$:

$$\frac{df}{dx}\approx\frac{f(x)-f(x-h)}{h}.$$

Sentraldifferansen bruker punkter på begge sider:

$$\frac{df}{dx}\approx\frac{f(x+h)-f(x-h)}{2h}.$$

Sentraldifferansen har vanligvis mindre tilnærmingsfeil enn de to ensidige differansene for samme steglengde.


In [4]:
def deriver_bakover(f, x, h=1e-5):
    return (f(x) - f(x - h)) / h

def deriver_sentralt(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2*h)

def f(x):
    return np.sin(x)

x0 = 1.0
eksakt = np.cos(x0)

for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    fram = deriver_framover(f, x0, h)
    bak = deriver_bakover(f, x0, h)
    sentral = deriver_sentralt(f, x0, h)
    print(f"h={h:g}: fram={fram:.8f}, bak={bak:.8f}, sentral={sentral:.8f}, eksakt={eksakt:.8f}")


h=0.1: fram=0.49736375, bak=0.58144075, sentral=0.53940225, eksakt=0.54030231
h=0.01: fram=0.53608598, bak=0.54450062, sentral=0.54029330, eksakt=0.54030231
h=0.001: fram=0.53988148, bak=0.54072295, sentral=0.54030222, eksakt=0.54030231
h=0.0001: fram=0.54026023, bak=0.54034438, sentral=0.54030230, eksakt=0.54030231


### Prøv selv

Sammenlikn de tre differansemetodene for ulike steglengder.

<iframe src="../../basthon/?from=examples/numerisk_derivasjon_metoder.py" width="100%" height="620" frameborder="0" title="Prøv selv: numerisk derivasjon" loading="lazy" allowfullscreen></iframe>

## Numerisk derivasjon av eksperimentelle data

For eksperimentelle data har vi ikke en funksjon $f(x)$ vi kan evaluere hvor vi vil. Målepunktene bestemmer avstanden mellom $x$-verdiene.

Dette gjør numerisk derivasjon svært nyttig, men også mer sårbar: **derivasjon forsterker ofte målestøy**, fordi små forskjeller mellom nabopunkter deles på et lite intervall.


In [5]:
import pandas as pd

data = pd.read_csv("../datafiler/titreringsdata.txt")
data.head()


,volum,pH
0,0.00,2.51
1,2.05,2.76
2,4.00,3.03
3,6.01,3.11
4,8.22,3.31


In [6]:
volum = data["volum"].to_numpy()
pH = data["pH"].to_numpy()

plt.plot(volum, pH, "o-")
plt.xlabel("Volum 0,10 M NaOH (mL)")
plt.ylabel("pH")
plt.title("Titrering av eddiksyre med NaOH")
plt.show()


### Hvor skal vi plassere den deriverte?

Hvis vi bruker to nabopunkter,

$$\frac{\Delta y}{\Delta x}=\frac{y_{i+1}-y_i}{x_{i+1}-x_i},$$

er dette den **gjennomsnittlige stigningen over intervallet** fra $x_i$ til $x_{i+1}$. Det mest naturlige stedet å plassere denne verdien er derfor midt i intervallet:

$$x_{\mathrm{midt}}=\frac{x_i+x_{i+1}}{2}.$$

Merk at dette er **gjennomsnittet av de to $x$-verdiene**, ikke halvparten av differansen $(x_{i+1}-x_i)/2$.

Hvis vi i stedet plasserer denne differansen ved $x_i$, forskyves den numeriske deriverte mot venstre i forhold til intervallet den faktisk beskriver.


In [7]:
dpH_dV = np.diff(pH) / np.diff(volum)
volum_midt = (volum[:-1] + volum[1:]) / 2

i_maks = np.argmax(dpH_dV)
V_eq = volum_midt[i_maks]

print(f"Største intervallstigning finnes rundt {V_eq:.2f} mL.")

plt.plot(volum, pH, "o-", label="Titrerkurve")
plt.xlabel("Volum 0,10 M NaOH (mL)")
plt.ylabel("pH")
plt.show()

plt.plot(volum_midt, dpH_dV, "o-", label=r"$\Delta\mathrm{pH}/\Delta V$")
plt.axvline(V_eq, linestyle="--")
plt.xlabel("Volum 0,10 M NaOH (mL)")
plt.ylabel(r"$\Delta\mathrm{pH}/\Delta V$")
plt.show()


Største intervallstigning finnes rundt 33.54 mL.


For disse dataene ligger den største stigningen mellom 33,52 og 33,56 mL. Midtpunktet er derfor **33,54 mL**. Det er et bedre estimat av plasseringen til denne intervallstigningen enn å bruke 33,52 mL direkte.

Dette betyr ikke at det sanne ekvivalenspunktet er kjent med hundredels milliliter. Målefrekvens, måleusikkerhet og valgt differansemetode begrenser presisjonen.

### `np.gradient`: en praktisk metode

Når vi har forstått differansene selv, kan NumPy gjøre arbeidet for oss. `np.gradient(y, x)` bruker sentrale differanser i indre punkter og ensidige differanser ved endepunktene. Den kan også bruke ujevnt fordelte $x$-verdier, slik vi har i titrerdataene.


In [8]:
gradient = np.gradient(pH, volum)
V_eq_gradient = volum[np.argmax(gradient)]

print(f"Maksimum med np.gradient ligger ved {V_eq_gradient:.2f} mL.")

plt.plot(volum, gradient, "o-")
plt.axvline(V_eq_gradient, linestyle="--")
plt.xlabel("Volum 0,10 M NaOH (mL)")
plt.ylabel(r"$d\mathrm{pH}/dV$")
plt.show()


Maksimum med np.gradient ligger ved 33.52 mL.


`np.diff`-metoden og `np.gradient` svarer ikke på helt identisk numerisk måte: Den første beregner stigningen **mellom** nabopunkter, mens `np.gradient` estimerer den deriverte **ved** de opprinnelige målepunktene. Derfor kan maksimum havne på litt ulike $x$-verdier.

I et virkelig analysearbeid bør vi ikke tolke en slik forskjell som kjemisk informasjon. Den viser at den numeriske metoden og oppløsningen i dataene påvirker estimatet.

```{admonition} Støy og glatting
:class: warning
Numerisk derivasjon kan gjøre tilfeldig målestøy mye tydeligere. Glatting kan noen ganger være nyttig, men den endrer også dataene. Derfor bør rådata, eventuell glatting og den deriverte alltid vurderes sammen.
```

## Kort oppsummering

- $f'(x)$ og $\frac{df}{dx}$ beskriver den samme deriverte.
- Numerisk derivasjon erstatter grenseverdien med et endelig steg.
- Framover-, bakover- og sentraldifferansen bruker ulike nabopunkter.
- Et for lite steg kan gi flyttallsproblemer; «mindre» er ikke alltid «bedre».
- For parvise differanser mellom målepunkter hører den beregnede stigningen naturlig til midtpunktet mellom $x_i$ og $x_{i+1}$.
- `np.gradient` er et praktisk verktøy når vi har forstått prinsippet.
- Derivasjon kan forsterke eksperimentell støy.


## Oppgaver

```{admonition} Oppgave 1 – numerisk og analytisk
:class: tip
Beregn $f'(1)$ numerisk og kontroller ved analytisk derivasjon for:

1. $f(x)=x^2-4x+5$
2. $f(x)=e^x$
3. $f(x)=\sqrt{\ln x}$
```

```{admonition} Oppgave 2 – feil som funksjon av steglengde
:class: tip
Sammenlikn framover- og sentraldifferansen for $f(x)=\sin x$ ved $x=1$. Lag et log-log-plott av absolutt feil for $h$ fra $10^{-1}$ til $10^{-15}$. Kommenter formen på kurvene.
```

```{admonition} Oppgave 3 – reaksjonsfart fra konsentrasjonsdata
:class: tip
Du har målt konsentrasjonen av A i en reaksjon:

`t = [0, 10, 20, 30, 40, 50]` s

`c = [1.00, 0.82, 0.68, 0.56, 0.47, 0.40]` mol/L

Beregn $\Delta c/\Delta t$ mellom hvert par av målinger og plott reaksjonsfarten $-\Delta c/\Delta t$ mot midtpunktene i tidsintervallene.
```

```{admonition} Oppgave 4 – titrering
:class: tip
Bruk titrerdataene i kapitlet.

1. Finn ekvivalenspunktet med `np.diff`.
2. Forklar hvorfor $x$-verdiene til den deriverte bør være $(x_i+x_{i+1})/2$.
3. Finn maksimum med `np.gradient`.
4. Sammenlikn estimatene og kommenter hvor mange sifre det er rimelig å rapportere.
```

```{admonition} Oppgave 5 – potensiell energi
:class: tip
En forenklet potensiell energi er $U(r)=r^{-12}-2r^{-6}$. Bruk sentraldifferansen til å finne omtrent hvor $dU/dr=0$. Kontroller ved å plotte $U(r)$.
```

```{admonition} Oppgave 6 – ujevnt fordelte data
:class: tip
Lag et lite datasett med ujevnt fordelte tidspunkter og en kjent funksjon. Sammenlikn en beregning som feilaktig antar konstant $\Delta t$ med `np.gradient(y, t)`.
```

```{admonition} Oppgave 7 – målestøy
:class: tip
Lag kunstige data fra $c(t)=e^{-0.1t}$ og legg til litt tilfeldig støy. Deriver både de støyfrie og de støyende dataene numerisk. Hva skjer med støyen?
```

```{admonition} Oppgave 8 – velg representasjon
:class: tip
Forklar med egne ord forskjellen mellom $f'(x)$, $df/dx$, $\Delta y/\Delta x$ og en numerisk tilnærming til $df/dx$.
```


## Video

<iframe width="890" height="500" src="https://www.youtube.com/embed/SDZTPkbZCi4" title="Numerisk derivasjon" frameborder="0" allowfullscreen></iframe>
